<div dir="rtl">
<h1>سه معماری را با مسیر اطلاعات تشخیص دهید</h1>
<p>درس 48 از 76 · Transformer چه فرقی با GPT دارد؟ · <code dir="ltr">42-families</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/42-families.html">📖 بازگشت به همین درس</a></p>
<p>Self-Attention دوطرفه، Self-Attention علّی و Cross-attention مستطیلی را با عدد مقایسه کنید.</p><p>پیش‌نیاز: Q/K/V، Mask و شکل ضرب Attention را بشناسید؛ منبع و هدف دو دنبالهٔ متفاوت‌اند.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>منبع ۵ موقعیت و خروجی فعلی ۳ موقعیت دارد. چرا بستن ستون‌های بعد از قطر در Cross-attention، بخشی از منبعِ ازپیش‌موجود را بی‌دلیل پنهان می‌کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
source = torch.tensor([[1.,0.],[0.,1.],[1.,1.],[-1.,0.],[0.,-1.]])
target = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
print('source/target lengths:',len(source),len(target))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع family_attention(q,k,v,kind) زوج (weights,output) برگرداند. kind برابر Encoder یا Decoder یا cross است. فقط در Decoder جدول مربعی را علّی کنید؛ در دو حالت دیگر همهٔ Keyها مجازند. q و k دو ویژگی مشترک دارند، ولی تعداد سطرهایشان می‌تواند متفاوت باشد.</p>
</div>

In [ ]:
def family_attention(q, k, v, kind):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = family_attention(target,source,source,'cross')
    if result is None: return False
    w,out = result
    assert w.shape == (3,5) and out.shape == (3,2)
    torch.testing.assert_close(w,(target@source.T/math.sqrt(2)).softmax(-1))
    torch.testing.assert_close(out,w@source)
    we,_ = family_attention(source,source,source,'encoder')
    assert we.shape == (5,5) and (we > 0).all()
    wd,_ = family_attention(target,target,target,'decoder')
    assert torch.count_nonzero(wd.triu(1)) == 0
    torch.testing.assert_close(wd.sum(-1),torch.ones(3))
    zq,zk,zv = torch.zeros(2,4),torch.zeros(3,4),torch.tensor([[1.],[2.],[6.]])
    torch.testing.assert_close(family_attention(zq,zk,zv,'cross')[1],torch.full((2,1),3.))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط طول منبع را از ۵ به ۴ کاهش دهید؛ Queryهای خروجی ثابت بمانند. تعداد سطرهای Cross-attention از هدف و تعداد ستون‌ها از منبع می‌آیند.</p>
</div>

In [ ]:
for length in (5,4):
    weights = (target@source[:length].T/math.sqrt(2)).softmax(-1)
    print('source length, weights/output shapes:',length,weights.shape,(weights@source[:length]).shape)

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>کد خراب Mask علّیِ هدف را به جدول Cross-attention تعمیم می‌دهد. تابع cross_weights(q,k) را اصلاح کنید؛ تمام منبع در این مثال از ابتدا موجود است.</p>
</div>

In [ ]:
scores = target@source.T/math.sqrt(2)
wrong = scores.masked_fill(~torch.ones(3,5,dtype=torch.bool).tril(),float('-inf')).softmax(-1)
print('wrong: source positions hidden from first target:',wrong[0])

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def cross_weights(q, k):
    # TODO
    return None

In [ ]:
def test_repair():
    result = cross_weights(target,source)
    if result is None: return False
    torch.testing.assert_close(result,(target@source.T/math.sqrt(2)).softmax(-1))
    assert (result > 0).all()
    torch.testing.assert_close(cross_weights(torch.zeros(2,3),torch.zeros(4,3)),torch.full((2,4),0.25))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>MiniGPT از حالت Decoder-only استفاده می‌کند: در هر بلوک Q/K/V از یک جریان‌اند و Mask علّی داریم. پروژه Encoder یا Cross-attention جدا ندارد؛ تابع کوچک این دفتر صرفاً برای مقایسهٔ خانواده‌هاست.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر کسی Transformer را مساوی GPT بداند، کدام تفاوتِ قابل مشاهده در این دفتر را از دست داده است؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/42-families.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/42-families.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>